In [0]:
#************************************************************************************************************************************
#*                                                                                                                                  *
#*   NOTEBOOK:     Auto_Run_EDW_Data_Load.                                                                                       *
#*                                                                                                                                  *
#*   DESCRIPTION:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT PARMS:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   INPUT FILES:                                                                                                                   *
#*                                                                                                                                  *
#*                                                                                                                                  *
#*   OUTPUT FILE:                                                                                                                   *
#*                                                                                                                                  *
#*   EXITS:       0 - success                                                                                                       *
#*                <> 0 - failure                                                                                                    *
#*                                                                                                                                  *
#************************************************************************************************************************************
#*                                                                                                                                  *
#*                                                 Modification Log                                                                 *
#*                                                                                                                                  *
#*    Date     CO                 Author              Description                                                                   *
#* ---------- ------------------  -----------------   ------------------------------------------------------------------------------*
#* 05/06/2025 000000000000000000  Ahmad Afzaal        Initial Release.                                                              *
#************************************************************************************************************************************


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("base_path", "s3a://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")
dbutils.widgets.text("log_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Ref")
dbutils.widgets.text("delta_table", "oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref") #temp tables Staging 
dbutils.widgets.text("copy_target_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/process/ETL_Ref")
dbutils.widgets.text("ctl_file_identifier", "ODM.EDW.VEN.REFERENCE.WEEKLY")
dbutils.widgets.text("columns_to_check_for_nulls", "SAK_RECIP")  # comma-separated

In [0]:
base_path = dbutils.widgets.get("base_path")
log_table = dbutils.widgets.get("log_table")
delta_table = dbutils.widgets.get("delta_table")
copy_target_path = dbutils.widgets.get("copy_target_path")
ctl_file_identifier = dbutils.widgets.get("ctl_file_identifier")
columns_to_check_for_nulls = dbutils.widgets.get("columns_to_check_for_nulls").split(",")


In [0]:
import os
import json
import re
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as f
from pyspark.dbutils import DBUtils
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import col
import pyspark.sql.functions as sqlf

# Initialize Spark session
spark = SparkSession.builder.appName("CTL-GZ-Reconciliation").getOrCreate()


# 2) Recursive file listing (rename loop var)
fi = []
def list_all_files(path):
    items = []
    for fi in dbutils.fs.ls(path):              
        if fi.isDir():
            items += list_all_files(fi.path)
        else:
            items.append(fi)
    return items

all_files = list_all_files(base_path)

top_folders = [d for d in dbutils.fs.ls(base_path) if d.isDir()]

folder_mods = []
for d in top_folders:
    try:
        # find the newest modificationTime among its children
        child_times = [
            datetime.fromtimestamp(f.modificationTime/1000, tz=timezone.utc)
            for f in dbutils.fs.ls(d.path)
        ]
        folder_mods.append((d.name, max(child_times)))
    except Exception:
        pass

# sort descending by that max timestamp
folder_mods.sort(key=lambda x: x[1], reverse=True)
last_folder_landed = folder_mods[0][0] if folder_mods else None
print(f"📂 Last folder landed under S3: {last_folder_landed}")


# 3) Find all CTL files by identifier

ctl_file_objs = [
    {
      "name": fi.name,
      "path": fi.path,
      "mod_time": datetime.fromtimestamp(fi.modificationTime/1000).replace(tzinfo=timezone.utc)
    }
    for fi in all_files                            
    if fi.name.endswith(".ctl") and ctl_file_identifier in fi.name
]

if not ctl_file_objs:
    print("❌ No .ctl files found matching identifier under base_path.")
    raise Exception("No CTLs to process")


# 4) Load reconciliation log

log_df = spark.read.table(log_table).select(
    "CTL_File", "Status", "Processed_Timestamp"
)

# 4a) Last successful CTL print
success_df = log_df.filter(
    (col("Status") == "SUCCESS") & 
    (col("CTL_File").contains(ctl_file_identifier))
)
if not success_df.rdd.isEmpty():
    last = success_df.orderBy(col("Processed_Timestamp").desc()).limit(1).collect()[0]
    print(f"✅ Last successfully processed CTL: {last['CTL_File']}  (on {last['Processed_Timestamp']})")
    # ── Locate which folder that CTL lived in ──
    last_ctl_name = last['CTL_File']
    last_success_folder = None

    # scan your all_files list for that name
    for fi in all_files:
        if fi.name == last_ctl_name:
            # drop the filename to get its parent folder
            # e.g. s3a://…/weekly/dt=20250605/ODM…ctl → dt=20250605
            last_success_folder = fi.path.rstrip("/").rsplit("/", 2)[-2]
            break

    print(f"📂 Folder of last successful CTL: {last_success_folder}")
    # (optionally send downstream)
    dbutils.jobs.taskValues.set(key="last_success_folder", value=last_success_folder)

else:
    print("ℹ️ No previously successful CTL in log.")

# 4b) Compute watermark of last success
if not success_df.rdd.isEmpty():
    last_success_time = success_df \
        .agg(sqlf.max(sqlf.col("Processed_Timestamp")).alias("max_ts")) \
        .collect()[0]["max_ts"]
else:
    last_success_time = datetime(1970,1,1, tzinfo=timezone.utc)

# 4c) Prepare sets for success vs. error
success_ctls = {r["CTL_File"] for r in success_df.collect()}
print(f"🎯 Number of successful CTLs matching '{ctl_file_identifier}': {success_df.count()}")
error_ctls = {
    r["CTL_File"] for r in log_df
    .filter((col("Status") != "SUCCESS") & col("CTL_File").contains(ctl_file_identifier))
    .collect()
}


# 5) Pick the one CTL to process

# retry failures up to 3 days old; new CTLs only if landed after last_success_time
retry_cutoff = datetime.now(timezone.utc) - timedelta(days=3)
eligible = []

for ctl in sorted(ctl_file_objs, key=lambda x: x["mod_time"], reverse=True):
    name = ctl["name"]
    mtime = ctl["mod_time"].replace(tzinfo=None)
    last_ok = last_success_time.replace(tzinfo=None)

    # a) retry errored CTLs in window
    if name in error_ctls and mtime >= retry_cutoff:
        eligible.append(ctl)
        break

    # b) new CTLs only if newer than last_success
    if name not in success_ctls and mtime > last_ok:
        eligible.append(ctl)
        break

if not eligible:
    print("✅ None: no new or retry-eligible CTLs found.")
    raise Exception("No CTLs to process")

to_process = eligible[0]
print(f"\n📄 CTL selected for processing: {to_process['name']}")
print(f"    → Landed: {to_process['mod_time']:%Y-%m-%d %H:%M:%S}")


# 6) Expose latest_ctl downstream

latest_ctl = dbutils.fs.ls(to_process["path"])[0]
dbutils.jobs.taskValues.set(
    key="ctl_received_date",
    value=to_process["mod_time"].strftime("%Y-%m-%d")
)


# 7) Read CTL and parse GZ expectations

ctl_df = spark.read.text(latest_ctl.path)
ctl_data = ctl_df.collect()

gz_expected_info = []
for row in ctl_data:
    parts = row[0].split('|')
    if len(parts) >= 2:
        gz_expected_info.append((parts[0], int(parts[1])))


# 8) Build GZ lookup across all files

gz_files_info = {
    fi.name: fi.path
    for fi in all_files                              
    if fi.name.endswith(".gz")
}

missing = [name for name, _ in gz_expected_info if name not in gz_files_info]
if missing:
    raise Exception(f"❌ Missing .gz files for this CTL: {missing}")

# Reconciliation
results = []
has_critical_failure = False
for gz_file_name, expected_count in gz_expected_info:
    matching_file_path = gz_files_info.get(gz_file_name)
    null_columns_failed = []
    error_count = None
    status = None
    actual_count = None

    if matching_file_path:
        try:
            # Check if file is non-empty
            if dbutils.fs.ls(matching_file_path):
                gz_df = spark.read.option("header", "true").option("inferSchema", "true").csv(matching_file_path)
                actual_count = gz_df.count()

                # Null check
                for col_name in columns_to_check_for_nulls:
                    if col_name in gz_df.columns:
                        null_count = gz_df.filter(col(col_name).isNull()).count()
                        if null_count > 0:
                            null_columns_failed.append(col_name)

                if null_columns_failed:
                    status = "NULL_VALUES_FOUND"
                    has_critical_failure = True
                    error_count = total_nulls
                elif actual_count != expected_count:
                    status = "Failed"
                    error_count = abs(actual_count - expected_count)
                else:
                    status = "SUCCESS"
            else:
                status = "EMPTY_FILE"
                actual_count = 0
                has_critical_failure = True
        except Exception as e:
            status = "CORRUPT"
            actual_count = None
            has_critical_failure = True
    else:
        status = "MISSING"
        actual_count = None
        has_critical_failure = True

    results.append(Row(
        CTL_File=latest_ctl.name,
        GZ_File=gz_file_name,
        Status=status,
        Row_Count_in_CTL=expected_count,
        Row_Count_in_GZ=actual_count,
        Null_Columns_Failed=",".join(null_columns_failed) if null_columns_failed else None,
        Number_of_Error_Records=error_count
    ))

    gz_files = [r['GZ_File'] for r in results if r['Status'] not in ["NULL_VALUES_FOUND", "MISSING", "CORRUPT"]]
    dbutils.jobs.taskValues.set(key="gz_file_list", value=gz_files)

# Display and log results
if results:
    result_df = spark.createDataFrame(results, schema="CTL_File STRING, GZ_File STRING, Status STRING, Row_Count_in_CTL INT, Row_Count_in_GZ INT, Null_Columns_Failed STRING, Number_of_Error_Records bigint")
    result_df = result_df.withColumn("Processed_Timestamp", f.current_timestamp())
    print("\nFinal Reconciliation Report:")
    result_df.show(truncate=False)

    # Append to Unity Catalog Delta table
    result_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)
if has_critical_failure:
    raise Exception("❌ One or more critical reconciliation failures detected. Stopping workflow.")

ctl_received_ts = to_process["mod_time"].strftime("%Y-%m-%d %H:%M:%S")
if results:
    schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received_by_APM", StringType(), True),
        StructField("Status", StringType(), True),
        StructField("Number_of_Records_Received", LongType(), True),
        StructField("Number_of_Records_Loaded", LongType(), True),
        StructField("Number_of_Error_Records", LongType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True)
    ])

    data = [
        (
            r['GZ_File'],  # File_Name
            ctl_received_ts,  # Date_Received_by_APM
            r[2],  # Status
            r[3],  # Number_of_Records_Received
            r[4],  # Number_of_Records_Loaded
            r['Number_of_Error_Records'],
            None,  # Start_Load_Date
            None   # End_Load_Date
        )
        for r in results if r['Status'] not in ["NULL_VALUES_FOUND", "MISSING", "CORRUPT"]
    ]

if not has_critical_failure:
    if any(r['Status'] == "Failed" for r in results):
        print("⚠️ Non-critical failure detected. Writing 'Failed' status and stopping workflow.")
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("append").saveAsTable(delta_table)
            print("📄 Wrote 'Failed' status to Delta table.")
        raise Exception("❌ Pre‐Validation encountered a non-critical failure. Stopping workflow.")
    else:
        # All statuses are success
        print("✅ No critical failure and all statuses are 'Success'. Proceeding with data load and file operations.")
        if data:
            df = spark.createDataFrame(data, schema=schema)
            df.write.format("delta").mode("append").saveAsTable(delta_table)
            print("📄 Uploaded to Delta table.")

        # Clean folder
        if not copy_target_path.endswith("/"):
            copy_target_path += "/"

        def folder_exists(path):
            try:
                _ = dbutils.fs.ls(path)
                return True
            except Exception:
                return False

        if folder_exists(copy_target_path):
            files_in_target = dbutils.fs.ls(copy_target_path)
            for f in files_in_target:
                try:
                    dbutils.fs.rm(f.path, recurse=True)
                except Exception as e:
                    print(f"❌ Failed to delete {f.path}: {e}")
            print(f"🧹 Cleared files inside {copy_target_path} (folder preserved)")
        else:
            print(f"📁 Folder {copy_target_path} does not exist, skipping deletion step")

        # Copy CTL
        try:
            dbutils.fs.cp(latest_ctl.path, copy_target_path + latest_ctl.name)
            print(f"📄 Copied CTL file: {latest_ctl.name}")
        except Exception as e:
            raise Exception(f"❌ Failed to copy CTL file: {e}")

        # Copy GZ files
        for gz_file_name, _ in gz_expected_info:
            gz_path = gz_files_info.get(gz_file_name)
            if gz_path:
                try:
                    dbutils.fs.cp(gz_path, copy_target_path + gz_file_name)
                    print(f"📦 Copied GZ file: {gz_file_name}")
                except Exception as e:
                    raise Exception(f"❌ Failed to copy GZ file: {gz_file_name}, error: {e}")

        # # Extract and return only the date portion from ctl_received_ts
            # dbutils.notebook.exit(json.dumps({"received_date": ctl_received_date}))
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            run_id = ctx.tags().get("runId")

            # Convert to string before storing
            if run_id:
                dbutils.jobs.taskValues.set(key="run_id", value=str(run_id))
            else:
                raise ValueError("❌ run_id not found in context. This notebook must be run as part of a job.")

else:
    print("⛔ Critical failure detected. Skipping all operations.")